In [2]:
import os

os.chdir('../')

In [3]:
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/gordillocarlosn/end-to-end-chest-cancer-clf.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "gordillocarlosn"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "1ceb80edbe44d9eef4f1827998225c19e89e47a1"

In [4]:
import tensorflow as tf

In [5]:
model = tf.keras.models.load_model('artifacts/training/model.h5')

Metal device set to: Apple M3 Max

systemMemory: 64.00 GB
maxCacheSize: 24.00 GB



In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [7]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json
import keras
import mlflow
import mlflow.tensorflow

keras.__version__ = tf.__version__
from urllib.parse import urlparse

In [8]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
        
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model = 'artifacts/training/model.h5',
            training_data = 'artifacts/data_ingestion/Chest-CT-Scan-data',
            all_params = self.params,
            mlflow_uri = "https://dagshub.com/gordillocarlosn/end-to-end-chest-cancer-clf.mlflow",
            params_batch_size = self.params.BATCH_SIZE,
            params_image_size = self.params.IMAGE_SIZE
        )
        return eval_config      

In [9]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
        
    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split = 0.3
        )
        
        dataflow_kwargs = dict(
            target_size = self.config.params_image_size[:-1],
            batch_size = self.config.params_batch_size,
            interpolation = 'bilinear'
        )
        
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(**datagenerator_kwargs)
        
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory = self.config.training_data,
            subset = 'validation',
            shuffle = False,
            **dataflow_kwargs
        )
    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    
    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()
    
    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path = Path('scores.json'), data = scores)
        
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics({"loss": self.score[0], "accuracy": self.score[1]})
            
            if tracking_url_type_store != 'file':
                # Register the model
                mlflow.tensorflow.log_model(self.model, 'model', registered_model_name = 'VGG16Model')
            else:
                mlflow.tensorflow.log_model(self.model, 'model')
            
    

In [10]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-03-09 00:01:38,079: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-03-09 00:01:38,081: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-09 00:01:38,082: INFO: common: created directory at: artifacts]
Found 102 images belonging to 2 classes.


2026-03-09 00:01:38.290685: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


7/7 [==============================] - 1s 76ms/step - loss: 9.1160e-08 - accuracy: 1.0000
[2026-03-09 00:01:39,020: INFO: common: json file saved at: scores.json]


2026/03/09 00:01:40 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-03-09 00:01:41,187: WARNING: save: Found untraced functions such as _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op while saving (showing 5 of 13). These functions will not be directly callable after loading.]
INFO:tensorflow:Assets written to: /var/folders/5h/jyxgwfc13dlb60p7w_pjj4_40000gn/T/tmp8vayrziz/model/data/model/assets
[2026-03-09 00:01:41,436: INFO: builder_impl: Assets written to: /var/folders/5h/jyxgwfc13dlb60p7w_pjj4_40000gn/T/tmp8vayrziz/model/data/model/assets]


/opt/homebrew/Caskroom/miniforge/base/envs/cancer/lib/python3.8/site-packages/_distutils_hack/__init__.py:32: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
Registered model 'VGG16Model' already exists. Creating a new version of this model...
2026/03/09 00:02:13 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: VGG16Model, version 5
Created version '5' of model 'VGG16Model'.
